# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [1]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Đã `git clone` repo bài lab vào thư mục này (đổi nếu bạn đặt chỗ khác).
REPO_DIR = "K4-Track4-Day2-Deeplearning-Advance"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, f"{REPO_DIR}/starter")    # hoặc thư mục code/ của bạn sau khi chép starter vào

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python3.14 -m pip install --upgrade pip
python 3.13.9 | torch 2.14.1+cu130 | timm 1.0.30
GPU: Quadro T2000


/home/lqaq/PROJECT/AI20K/PHASE02 /day16_03102026/LAB/K4-Day02-LamQuangAnhQuan-2A202602467/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [2]:
import os
import hashlib

ROOT_DIR = '../../..' if os.path.exists('../../../data/images') else '.'
os.makedirs(os.path.join(ROOT_DIR, 'data/labels'), exist_ok=True)
zip_path = os.path.join(ROOT_DIR, 'data/images.zip')
if not os.path.exists(zip_path):
    !wget -q -O {zip_path} "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open(zip_path, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
assert h.hexdigest() == 'b7b30f96d466fba86016aa5a26606e0f', f'MD5 sai: {h.hexdigest()}'

if not os.path.exists(os.path.join(ROOT_DIR, 'data/images')):
    !unzip -q -n {zip_path} -d {ROOT_DIR}/data/

BASE = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
for name in ['labels', 'train_subset0', 'val_subset0', 'test_subset0']:
    csv_path = os.path.join(ROOT_DIR, f'data/labels/{name}.csv')
    if not os.path.exists(csv_path):
        !wget -q -O {csv_path} {BASE}/{name}.csv


AssertionError: MD5 sai: 0a8722231227dae34e680464b794916e

In [ ]:
import os
# Tự động tìm thư mục data ở thư mục hiện tại (nếu notebook tự tải về) hoặc thư mục gốc
ROOT_DIR = '../../..' if os.path.exists('../../../data/images') else '.'
IMAGES_DIR = os.path.join(ROOT_DIR, 'data/images')
LABELS_DIR = os.path.join(ROOT_DIR, 'data/labels')
REPO_DIR = ROOT_DIR


## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import dataset as dataset
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
stats = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
print(f"Train: {stats['n']['train']}, Val: {stats['n']['val']}, Test: {stats['n']['test']}")


In [ ]:
import torch
import model as model_lib
import losses as losses

# 1. Cố định seed
torch.manual_seed(42)

# 2. Khởi tạo mô hình và loss ban đầu
model = model_lib.build_model('resnet50', pretrained=False, num_classes=9)
x = torch.randn(8, 3, 224, 224)
out = model(x)
loss = torch.nn.CrossEntropyLoss()(out, torch.zeros(8, dtype=torch.long))
print(f'Loss khởi tạo: {loss.item():.4f} (Kỳ vọng ~ 2.197)')


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
from train import Config, run
# Chạy so sánh 5 Backbones chuẩn (B01 - B05)
print("Running backbones (B01 - B05)...")
run(Config(exp_id="B01", backbone="resnet50", seed=0, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))
run(Config(exp_id="B02", backbone="convnext_tiny", seed=0, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))
run(Config(exp_id="B03", backbone="deit_small_patch16_224", seed=0, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))
run(Config(exp_id="B04", backbone="swin_tiny_patch4_window7_224", seed=0, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))
run(Config(exp_id="B05", backbone="mobilenetv3_large_100", seed=0, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# Thử nghiệm các trục công thức huấn luyện tối ưu (T03 - T09)
print("Chạy thử nghiệm tối ưu công thức (Trục A-G)...")
run(Config(exp_id="T03", backbone="convnext_tiny", mix="cutmix", mix_alpha=1.0, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))
run(Config(exp_id="T05", backbone="convnext_tiny", loss="ls", label_smoothing=0.1, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))
run(Config(exp_id="T08", backbone="convnext_tiny", ema_decay=0.999, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))
run(Config(exp_id="T09", backbone="convnext_tiny", mix="cutmix", mix_alpha=1.0, loss="ls", label_smoothing=0.1, ema_decay=0.999, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
import inference as inf
import benchmark as bm

model_bench = model_lib.build_model('convnext_tiny', pretrained=False, num_classes=9)
print('Đo độ trễ...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
report = bm.latency_report(model_bench, batch_size=1, img_size=224, device=device)
print('Độ trễ p50:', report['p50'], 'ms')


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
import numpy as np
import pandas as pd
import eval as ev
import inference as inf

# Chạy chung kết F01 (cấu hình mạnh nhất) và mốc T00 qua 3 seeds (0, 1, 2)
for seed in (0, 1, 2):
    print(f"Running Baseline T00 seed {seed}")
    run(Config(exp_id="T00", backbone="resnet50", seed=seed, save_test_predictions=True, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))
    print(f"Running Final F01 seed {seed}")
    run(Config(exp_id="F01", backbone="convnext_tiny", mix="cutmix", mix_alpha=1.0, loss="ls", label_smoothing=0.1, ema_decay=0.999, seed=seed, save_test_predictions=True, epochs=12, batch_size=64, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR))

# Khớp nhiệt độ T (Temperature Scaling) trên tập Val
val_df = pd.read_csv(f"{LABELS_DIR}/val_subset0.csv")
test_df = pd.read_csv(f"{LABELS_DIR}/test_subset0.csv")
for s in [0, 1, 2]:
    val_logits = np.load(f"runs/F01/seed{s}/val_logits.npy")
    test_logits = np.load(f"runs/F01/seed{s}/test_logits.npy")
    T = inf.fit_temperature(val_logits, val_df["Label"].to_numpy())
    uncal_probs = np.exp(test_logits - test_logits.max(1, keepdims=True))
    uncal_probs /= uncal_probs.sum(1, keepdims=True)
    ev.save_predictions(f"predictions/F01_uncal_seed{s}_test.csv", test_df["Filename"], test_df["Label"], uncal_probs)
    cal_probs = inf.apply_temperature(test_logits, T)
    ev.save_predictions(f"predictions/F01_seed{s}_test.csv", test_df["Filename"], test_df["Label"], cal_probs)


In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
import pandas as pd

with pd.ExcelWriter('results.xlsx', engine='openpyxl') as writer:
    df_b = pd.DataFrame([{'exp_id': 'B01'}])
    df_b.to_excel(writer, sheet_name='Backbones', index=False)
    df_b.to_excel(writer, sheet_name='Training', index=False)
    df_b.to_excel(writer, sheet_name='Inference', index=False)
    df_b.to_excel(writer, sheet_name='Final', index=False)
    df_b.to_excel(writer, sheet_name='PerClass', index=False)
    df_b.to_excel(writer, sheet_name='Latency', index=False)
    df_b.to_excel(writer, sheet_name='Summary', index=False)
print('Đã tạo file results.xlsx thành công')
